In [ ]:
from model import RidershipModel

m = RidershipModel()
m.load_data()
m.fit()

In [ ]:
import os
import pandas as pd

PREDICTIONS_PATH = "data/predictions.parquet"

if os.path.exists(PREDICTIONS_PATH):
    results = pd.read_parquet(PREDICTIONS_PATH)
    print(f"Loaded predictions from {PREDICTIONS_PATH}")
else:
    results = m.predict()
    results.to_parquet(PREDICTIONS_PATH, index=False)
    print(f"Saved predictions to {PREDICTIONS_PATH}")

In [ ]:
import plotly.graph_objects as go

# Order stations by 2024 ridership (busiest first)
station_order = (
    m.train.groupby("station_complex")["ridership"]
    .sum()
    .loc[lambda s: s.index.isin(results["station_complex"].unique())]
    .sort_values(ascending=False)
    .index.tolist()
)

# Precompute per-station traces
station_traces = {}
for station in station_order:
    train_df = m.train[m.train["station_complex"] == station].sort_values("date")
    pred_df = results[results["station_complex"] == station].sort_values("ds")
    station_traces[station] = (train_df, pred_df)

default = station_order[0]
train_df, pred_df = station_traces[default]

fwd = pred_df["ds"].dt.strftime("%Y-%m-%d").tolist()
rev = fwd[::-1]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=train_df["date"].dt.strftime("%Y-%m-%d"),
    y=train_df["ridership"],
    mode="lines", name="2024 actual",
    line=dict(color="#4C78A8"),
))

fig.add_trace(go.Scatter(
    x=fwd + rev,
    y=pred_df["yhat_upper"].tolist() + pred_df["yhat_lower"].tolist()[::-1],
    fill="toself", fillcolor="rgba(255,127,14,0.15)",
    line=dict(color="rgba(0,0,0,0)"),
    name="forecast interval",
    hoverinfo="skip",
))

fig.add_trace(go.Scatter(
    x=fwd, y=pred_df["yhat"],
    mode="lines", name="2025 forecast",
    line=dict(color="#F58518", dash="dash"),
))

fig.add_trace(go.Scatter(
    x=fwd, y=pred_df["actual"],
    mode="lines", name="2025 actual",
    line=dict(color="#54A24B"),
))

buttons = []
for station in station_order:
    td, pr = station_traces[station]
    f = pr["ds"].dt.strftime("%Y-%m-%d").tolist()
    r = f[::-1]
    buttons.append(dict(
        label=station,
        method="update",
        args=[
            {
                "x": [
                    td["date"].dt.strftime("%Y-%m-%d").tolist(),
                    f + r,
                    f,
                    f,
                ],
                "y": [
                    td["ridership"].tolist(),
                    pr["yhat_upper"].tolist() + pr["yhat_lower"].tolist()[::-1],
                    pr["yhat"].tolist(),
                    pr["actual"].tolist(),
                ],
            },
            {"title": f"Daily Ridership — {station}"},
        ],
    ))

fig.update_layout(
    title=f"Daily Ridership — {default}",
    xaxis_title="Date",
    yaxis_title="Daily Ridership",
    height=580,
    hovermode="x unified",
    updatemenus=[dict(
        type="dropdown",
        buttons=buttons,
        x=0, xanchor="left",
        y=1.18, yanchor="top",
        bgcolor="white",
        bordercolor="#cccccc",
    )],
    margin=dict(t=120),
    shapes=[dict(
        type="line",
        x0="2025-01-01", x1="2025-01-01",
        y0=0, y1=1, yref="paper",
        line=dict(color="gray", dash="dot", width=1),
    )],
    annotations=[dict(
        x="2025-01-01", y=1, yref="paper",
        text="2025 →", showarrow=False,
        xanchor="left", yanchor="bottom",
        font=dict(color="gray", size=11),
    )],
)
fig.show()